In [1]:
import sys
import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Python: 3.10.12 (main, Jun 11 2023, 05:26:28) [GCC 11.4.0]
PyTorch: 2.0.1+cu118
CUDA: True
GPU: Tesla T4


In [4]:
from pathlib import Path
import subprocess

project_dir = Path.cwd() / "error-geometry-routing-frozen"

if not project_dir.exists():
    subprocess.run([
        "git", "clone",
        "--branch", "geometry",
        "--single-branch",
        "https://github.com/Ottobiss/error-geometry-routing.git",
        str(project_dir),
    ], check=True)

print("Папка проекта:", project_dir)

Папка проекта: /home/jupyter/project/error-geometry-routing-frozen


In [5]:
result = subprocess.run(
    ["git", "branch", "--list", "frozen-router-f2-f3"],
    cwd=project_dir,
    capture_output=True,
    text=True,
    check=True,
)

if not result.stdout.strip():
    subprocess.run(
        ["git", "switch", "-c", "frozen-router-f2-f3"],
        cwd=project_dir,
        check=True,
    )
else:
    subprocess.run(
        ["git", "switch", "frozen-router-f2-f3"],
        cwd=project_dir,
        check=True,
    )

Switched to a new branch 'frozen-router-f2-f3'


In [6]:
patch_path = Path.cwd() / "frozen-router-f2-f3.patch"

if not (project_dir / "routing/frozen/preflight.py").exists():
    subprocess.run(
        ["git", "apply", "--check", str(patch_path)],
        cwd=project_dir,
        check=True,
    )
    subprocess.run(
        ["git", "apply", str(patch_path)],
        cwd=project_dir,
        check=True,
    )

print("Код селектора подготовлен")

Код селектора подготовлен


In [7]:
logs_dir = project_dir / "logs"
logs_dir.mkdir(exist_ok=True)

result = subprocess.run(
    [sys.executable, "-m", "routing.frozen.preflight"],
    cwd=project_dir,
    capture_output=True,
    text=True,
)

output = result.stdout + result.stderr
(logs_dir / "preflight.txt").write_text(output, encoding="utf-8")

print(output)
print("Код завершения:", result.returncode)
print("Файл:", logs_dir / "preflight.txt")

{
  "python": "3.10.12 (main, Jun 11 2023, 05:26:28) [GCC 11.4.0]",
  "modules": {
    "torch": true,
    "transformers": false,
    "numpy": true,
    "scipy": true,
    "PIL": true,
    "cityscapesscripts": false
  },
  "nvidia_smi": "/usr/bin/nvidia-smi",
  "torch": "2.0.1+cu118",
  "cuda_available": true,
  "gpu": "Tesla T4",
  "total_memory_bytes": 15835660288
}

Код завершения: 1
Файл: /home/jupyter/project/error-geometry-routing-frozen/logs/preflight.txt


In [9]:
%pip install transformers==4.46.3 cityscapesscripts

Defaulting to user installation because normal site-packages is not writeable
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 23.8 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 22.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 32.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 28.4 MB/s  0:00:00
  Created wheel for typing: filename=typing-3.7.4.3-py3-none-any.whl size=26398 sha256=cb2cd8103194e786edfeb721940257a949e7a0f80c393aa5ac36d2cd54967ae0
  Stored in directory: /tmp/xdg_cache/pip/wheels/7c/d0/9e/1f26ebb66d9e1732e4098bc5a6c2d91f6c9a529838f0284890
Successfully built typing
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10 [transformers] [transformers]ipts]

[notice] A new release of pip is available: 23.0.1 -> 26.2.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [1]:
import sys
import subprocess
from pathlib import Path

project_dir = Path("/home/jupyter/project/error-geometry-routing-frozen")
logs_dir = project_dir / "logs"
logs_dir.mkdir(exist_ok=True)

result = subprocess.run(
    [sys.executable, "-m", "routing.frozen.preflight"],
    cwd=project_dir,
    capture_output=True,
    text=True,
)

output = result.stdout + result.stderr
(logs_dir / "preflight.txt").write_text(output, encoding="utf-8")

print(output)
print("Код завершения:", result.returncode)

{
  "python": "3.10.12 (main, Jun 11 2023, 05:26:28) [GCC 11.4.0]",
  "modules": {
    "torch": true,
    "transformers": true,
    "numpy": true,
    "scipy": true,
    "PIL": true,
    "cityscapesscripts": true
  },
  "nvidia_smi": "/usr/bin/nvidia-smi",
  "torch": "2.0.1+cu118",
  "cuda_available": true,
  "gpu": "Tesla T4",
  "total_memory_bytes": 15835660288,
  "transformers": "4.46.3"
}
/home/jupyter/.local/lib/python3.10/site-packages/transformers/utils/hub.py:128: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(

Код завершения: 0


In [2]:
result = subprocess.run(
    [sys.executable, "-m", "unittest", "discover",
     "-s", "tests", "-p", "test_frozen_router.py", "-v"],
    cwd=project_dir,
    capture_output=True,
    text=True,
)

output = result.stdout + result.stderr
(logs_dir / "frozen_router_tests.txt").write_text(output, encoding="utf-8")

print(output)
print("Код завершения:", result.returncode)

test_cache_policy_mismatch (test_frozen_router.FrozenCPU) ... ok
test_cli_train_and_calibrate_synthetic (test_frozen_router.FrozenCPU) ... ok
test_component_target_matches_signed_pixels (test_frozen_router.FrozenCPU) ... ok
test_components_eight_connected (test_frozen_router.FrozenCPU) ... ok
test_geometry_cli_and_selector_synthetic (test_frozen_router.FrozenCPU) ... ok
test_geometry_missing_classes_finite_features (test_frozen_router.FrozenCPU) ... ok
test_ignore_confusion (test_frozen_router.FrozenCPU) ... ok
test_incomplete_cache_rejected (test_frozen_router.FrozenCPU) ... ok
test_mahalanobis_matches_direct_inverse (test_frozen_router.FrozenCPU) ... ok
test_merge_zero_and_all (test_frozen_router.FrozenCPU) ... ok
test_purity_and_ignore (test_frozen_router.FrozenCPU) ... ok
test_ridge_rejects_nonfinite (test_frozen_router.FrozenCPU) ... ok
test_ridge_weighted_fit (test_frozen_router.FrozenCPU) ... ok
test_split_overlap (test_frozen_router.FrozenCPU) ... ok
test_window_origin_boundari

In [4]:
from pathlib import Path
import subprocess
import sys

project_dir = Path("/home/jupyter/project/error-geometry-routing-frozen")

result = subprocess.run(
    [sys.executable, "-c",
     "from routing.frozen.backbone import FrozenB0, load_b2; "
     "print('Импорт моделей: OK')"],
    cwd=project_dir,
    capture_output=True,
    text=True,
)

print(result.stdout)
print(result.stderr)
print("Код завершения:", result.returncode)

for root in [
    project_dir / "data/cityscapes",
    Path("/home/jupyter/project/data/cityscapes"),
    Path("/data/cityscapes"),
]:
    print(root, "—", "найден" if root.exists() else "нет")

Импорт моделей: OK

/home/jupyter/.local/lib/python3.10/site-packages/transformers/utils/hub.py:128: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
2026-10-25 13:11:24.952681: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-10-25 13:11:24.999964: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-10-25 13:11:25.825029: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
/usr/local

In [5]:
%pip install gdown==6.3.0

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.0.1 -> 26.2.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [6]:
import sys
import subprocess
import json

drive_url = (
    "https://drive.google.com/drive/folders/"
    "1O8_9tp_lj4ZHjKisqcUszGtdvFvXQp74"
)

result = subprocess.run(
    [sys.executable, "-m", "gdown", drive_url, "--json"],
    capture_output=True,
    text=True,
)

if result.returncode == 0:
    drive_files = json.loads(result.stdout)
    print("Всего файлов:", len(drive_files))
    for item in drive_files[:30]:
        print(item["path"])
else:
    print(result.stderr)

Всего файлов: 3
gtFine_trainvaltest.zip
test.zip
val.zip


In [7]:
from pathlib import Path
import subprocess
import sys

project_dir = Path("/home/jupyter/project/error-geometry-routing-frozen")
archive_dir = project_dir / "data/cityscapes_archives"
archive_dir.mkdir(parents=True, exist_ok=True)

for name in ["val.zip", "gtFine_trainvaltest.zip"]:
    matches = [
        item for item in drive_files
        if Path(item["path"]).name == name
    ]
    if len(matches) != 1:
        raise ValueError(f"Не удалось однозначно найти {name}")

    destination = archive_dir / name

    subprocess.run(
        [
            sys.executable, "-m", "gdown",
            matches[0]["url"],
            "--output", str(destination),
            "--continue",
        ],
        check=True,
    )

    print(name, "—", round(destination.stat().st_size / 1024**2, 1), "МиБ")

Downloading...
From (original): https://drive.google.com/uc?id=1aErjLEt25oJEX48W-zxHwITEvNAeKxGJ
From (redirected): https://drive.google.com/uc?id=1aErjLEt25oJEX48W-zxHwITEvNAeKxGJ&confirm=t&uuid=ea71fcc3-3c96-4cab-94b5-8fde7be47687
To: /home/jupyter/project/error-geometry-routing-frozen/data/cityscapes_archives/val.zip
100%|██████████| 1.18G/1.18G [00:09<00:00, 118MB/s] 


val.zip — 1127.1 МиБ


Using cookies from /home/jupyter/.cache/gdown/cookies.txt
Downloading...
From (original): https://drive.google.com/uc?id=1AxGkcacU_SZXPSjxUtzKgT3XSZM032bk
From (redirected): https://drive.google.com/uc?id=1AxGkcacU_SZXPSjxUtzKgT3XSZM032bk&confirm=t&uuid=634670d9-5317-4a37-b0af-6f93b5b3e751
To: /home/jupyter/project/error-geometry-routing-frozen/data/cityscapes_archives/gtFine_trainvaltest.zip
100%|██████████| 253M/253M [00:07<00:00, 35.5MB/s] 


gtFine_trainvaltest.zip — 240.9 МиБ


In [8]:
import zipfile

for name in ["val.zip", "gtFine_trainvaltest.zip"]:
    with zipfile.ZipFile(archive_dir / name) as archive:
        names = [n for n in archive.namelist() if not n.endswith("/")]

        print(f"\n{name}: {len(names)} файлов")
        print("Первые 10:")
        for entry in names[:10]:
            print(" ", entry)

        print(
            "Изображений leftImg8bit:",
            sum(n.endswith("_leftImg8bit.png") for n in names),
        )
        print(
            "Масок labelIds:",
            sum(n.endswith("_gtFine_labelIds.png") for n in names),
        )


val.zip: 500 файлов
Первые 10:
  val/frankfurt/frankfurt_000000_000294_leftImg8bit.png
  val/frankfurt/frankfurt_000000_000576_leftImg8bit.png
  val/frankfurt/frankfurt_000000_001016_leftImg8bit.png
  val/frankfurt/frankfurt_000000_001236_leftImg8bit.png
  val/frankfurt/frankfurt_000000_001751_leftImg8bit.png
  val/frankfurt/frankfurt_000000_002196_leftImg8bit.png
  val/frankfurt/frankfurt_000000_002963_leftImg8bit.png
  val/frankfurt/frankfurt_000000_003025_leftImg8bit.png
  val/frankfurt/frankfurt_000000_003357_leftImg8bit.png
  val/frankfurt/frankfurt_000000_003920_leftImg8bit.png
Изображений leftImg8bit: 500
Масок labelIds: 0

gtFine_trainvaltest.zip: 20002 файлов
Первые 10:
  README
  license.txt
  gtFine/train/jena/jena_000020_000019_gtFine_labelIds.png
  gtFine/train/jena/jena_000092_000019_gtFine_polygons.json
  gtFine/train/jena/jena_000061_000019_gtFine_polygons.json
  gtFine/train/jena/jena_000104_000019_gtFine_polygons.json
  gtFine/train/jena/jena_000064_000019_gtFine_lab

In [9]:
from pathlib import Path, PurePosixPath
import zipfile
import shutil

project_dir = Path("/home/jupyter/project/error-geometry-routing-frozen")
archive_dir = project_dir / "data/cityscapes_archives"
data_root = project_dir / "data/cityscapes"


def copy_entry(archive, entry, destination):
    destination.parent.mkdir(parents=True, exist_ok=True)
    with archive.open(entry) as source, destination.open("wb") as target:
        shutil.copyfileobj(source, target)


image_count = 0
with zipfile.ZipFile(archive_dir / "val.zip") as archive:
    for entry in archive.infolist():
        parts = PurePosixPath(entry.filename).parts

        if (
            len(parts) == 3
            and parts[0] == "val"
            and ".." not in parts
            and parts[-1].endswith("_leftImg8bit.png")
        ):
            destination = data_root / "leftImg8bit" / Path(*parts)
            copy_entry(archive, entry, destination)
            image_count += 1

label_count = 0
with zipfile.ZipFile(archive_dir / "gtFine_trainvaltest.zip") as archive:
    for entry in archive.infolist():
        parts = PurePosixPath(entry.filename).parts

        if (
            len(parts) == 4
            and parts[:2] == ("gtFine", "val")
            and ".." not in parts
            and parts[-1].endswith("_gtFine_labelIds.png")
        ):
            destination = data_root / Path(*parts)
            copy_entry(archive, entry, destination)
            label_count += 1

    if "license.txt" in archive.namelist():
        copy_entry(
            archive,
            archive.getinfo("license.txt"),
            data_root / "license.txt",
        )

print("Изображений распаковано:", image_count)
print("Масок распаковано:", label_count)

assert image_count == 500
assert label_count == 500

Изображений распаковано: 500
Масок распаковано: 500


In [10]:
images = sorted(
    (data_root / "leftImg8bit/val").rglob("*_leftImg8bit.png")
)

missing = []
for image in images:
    image_id = image.name.removesuffix("_leftImg8bit.png")
    label = (
        data_root / "gtFine/val" / image.parent.name
        / f"{image_id}_gtFine_labelIds.png"
    )
    if not label.exists():
        missing.append(str(label))

print("Изображений:", len(images))
print("Недостающих масок:", len(missing))

assert len(images) == 500
assert not missing, missing[:5]

print("Данные готовы:", data_root)

Изображений: 500
Недостающих масок: 0
Данные готовы: /home/jupyter/project/error-geometry-routing-frozen/data/cityscapes


In [11]:
import sys
import subprocess
from datetime import datetime

available_ids = {
    image.name.removesuffix("_leftImg8bit.png")
    for image in images
}

p0_path = project_dir / "splits/cityscapes_p0.txt"
p0_ids = p0_path.read_text().split()
smoke_ids = [image_id for image_id in p0_ids if image_id in available_ids][:2]

assert len(smoke_ids) == 2, "Не найдены два кадра из P0"

results_dir = project_dir / "results/frozen"
results_dir.mkdir(parents=True, exist_ok=True)

ids_path = results_dir / "smoke_ids.txt"
ids_path.write_text("\n".join(smoke_ids) + "\n")

out_dir = results_dir / f"smoke_{datetime.now():%Y%m%d_%H%M%S}"

logs_dir = project_dir / "logs"
logs_dir.mkdir(exist_ok=True)
log_path = logs_dir / f"{out_dir.name}.txt"

command = [
    sys.executable, "-u", "-m", "routing.frozen.export",
    "--dataset", "cityscapes",
    "--root", str(data_root),
    "--split", "val",
    "--ids", str(ids_path),
    "--out", str(out_dir),
    "--layers", "F2,F3,F4,C",
    "--max-batch", "1",
    "--workers", "0",
]

print("Кадры:", smoke_ids)
print("Результаты:", out_dir)
print("Первый запуск может скачивать готовые веса B0/B2.")

with log_path.open("w") as log:
    process = subprocess.Popen(
        command,
        cwd=project_dir,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    for line in process.stdout:
        print(line, end="", flush=True)
        log.write(line)
        log.flush()

    returncode = process.wait()

print("\nКод завершения:", returncode)
print("Лог:", log_path)

Кадры: ['frankfurt_000000_000294', 'frankfurt_000000_001016']
Результаты: /home/jupyter/project/error-geometry-routing-frozen/results/frozen/smoke_20261025_131838
Первый запуск может скачивать готовые веса B0/B2.
/home/jupyter/.local/lib/python3.10/site-packages/transformers/utils/hub.py:128: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
2026-10-25 13:18:43.524275: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-10-25 13:18:43.572197: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operation

In [12]:
from pathlib import Path
import json
import hashlib
import numpy as np

smoke_dir = Path(
    "/home/jupyter/project/error-geometry-routing-frozen/"
    "results/frozen/smoke_20261025_131838"
)

meta = json.loads((smoke_dir / "manifest.json").read_text())
assert meta["complete"], "Кэш не завершён"

expected_channels = {
    "F2": 64,
    "F3": 160,
    "F4": 256,
    "C": 256,
}

for image_id in meta["images"]:
    filename = hashlib.sha256(image_id.encode()).hexdigest()[:24] + ".npz"

    with np.load(smoke_dir / filename) as data:
        print("\nКадр:", image_id)

        for layer, channels in expected_channels.items():
            feature = data[layer]
            assert feature.ndim == 3
            assert feature.shape[0] == channels
            assert np.isfinite(feature).all(), f"NaN/Inf в {layer}"
            print(layer, feature.shape, feature.dtype)

        b0, b2, gt = data["b0"], data["b2"], data["gt"]
        valid = gt != 255

        assert b0.shape == b2.shape == gt.shape
        assert np.isin(b0, np.arange(19)).all()
        assert np.isin(b2, np.arange(19)).all()
        assert np.array_equal(data["valid"], valid)
        assert np.isfinite(data["confidence"]).all()
        assert np.isfinite(data["entropy"]).all()

        expected_gain = (
            (b2 == gt).astype(np.int8)
            - (b0 == gt).astype(np.int8)
        ) * valid

        assert np.array_equal(data["gain"], expected_gain)

        print("Маски:", b0.shape)
        print("Исправлено B2:", int((expected_gain == 1).sum()))
        print("Испорчено B2:", int((expected_gain == -1).sum()))

print("\nПроверка кэша: OK")
print(
    "Размер smoke-кэша:",
    round(sum(p.stat().st_size for p in smoke_dir.glob("*.npz")) / 1024**2, 1),
    "МиБ",
)


Кадр: frankfurt_000000_000294
F2 (64, 64, 128) float16
F3 (160, 32, 64) float16
F4 (256, 16, 32) float16
C (256, 128, 256) float16
Маски: (512, 1024)
Исправлено B2: 8410
Испорчено B2: 1741

Кадр: frankfurt_000000_001016
F2 (64, 64, 128) float16
F3 (160, 32, 64) float16
F4 (256, 16, 32) float16
C (256, 128, 256) float16
Маски: (512, 1024)
Исправлено B2: 6929
Испорчено B2: 1818

Проверка кэша: OK
Размер smoke-кэша: 23.5 МиБ


In [13]:
import shutil

usage = shutil.disk_usage("/home/jupyter/project")
print("Свободно:", round(usage.free / 1024**3, 1), "ГиБ")

Свободно: 7.3 ГиБ


In [14]:
from pathlib import Path
import numpy as np
import json

project_dir = Path("/home/jupyter/project/error-geometry-routing-frozen")
data_root = project_dir / "data/cityscapes"

p0 = sorted((project_dir / "splits/cityscapes_p0.txt").read_text().split())
p1 = sorted((project_dir / "splits/cityscapes_p1.txt").read_text().split())

assert len(p0) == len(set(p0))
assert len(p1) == len(set(p1))
assert not set(p0) & set(p1)


def sequence_id(image_id):
    return "_".join(image_id.split("_")[:2])


groups = sorted({sequence_id(i) for i in p0})
assert len(groups) >= 2

group_sizes = {
    group: sum(sequence_id(i) == group for i in p0)
    for group in groups
}

# Ищем примерно 20% P0 для validation, не разделяя последовательности.
rng = np.random.default_rng(42)
target = round(0.2 * len(p0))
best = None

for _ in range(1000):
    order = rng.permutation(groups).tolist()
    cumulative = np.cumsum([group_sizes[g] for g in order])
    cut = int(np.argmin(abs(cumulative[:-1] - target))) + 1
    candidate = set(order[:cut])
    difference = abs(int(cumulative[cut - 1]) - target)

    if best is None or difference < best[0]:
        best = (difference, candidate)

val_groups = best[1]
train_ids = [i for i in p0 if sequence_id(i) not in val_groups]
val_ids = [i for i in p0 if sequence_id(i) in val_groups]
test_ids = p1

assert train_ids and val_ids
assert not set(train_ids) & set(val_ids)
assert not {sequence_id(i) for i in train_ids} & {
    sequence_id(i) for i in val_ids
}

split_dir = project_dir / "splits/frozen_router_v1"
split_dir.mkdir(parents=True, exist_ok=True)

splits = {
    "train": train_ids,
    "val": val_ids,
    "test": test_ids,
}

for name, ids in splits.items():
    path = split_dir / f"{name}.txt"
    text = "\n".join(ids) + "\n"

    if path.exists() and path.read_text() != text:
        raise RuntimeError(f"Уже существует другое разбиение: {path}")

    path.write_text(text)

    cities = {}
    for image_id in ids:
        city = image_id.split("_")[0]
        cities[city] = cities.get(city, 0) + 1

    print(name, "кадров:", len(ids), "по городам:", cities)

print("Разбиения сохранены:", split_dir)

train кадров: 240 по городам: {'frankfurt': 126, 'lindau': 25, 'munster': 89}
val кадров: 60 по городам: {'frankfurt': 33, 'lindau': 6, 'munster': 21}
test кадров: 200 по городам: {'frankfurt': 108, 'lindau': 28, 'munster': 64}
Разбиения сохранены: /home/jupyter/project/error-geometry-routing-frozen/splits/frozen_router_v1


In [15]:
import sys
import subprocess
from datetime import datetime

run_name = f"routing_v1_{datetime.now():%Y%m%d_%H%M%S}"
cache_root = project_dir / "results/frozen" / run_name

logs_dir = project_dir / "logs"
logs_dir.mkdir(exist_ok=True)

# Сохраняем пути, чтобы не потерять их после перезапуска ядра.
plan = {
    "cache_root": str(cache_root),
    "split_dir": str(split_dir),
    "data_root": str(data_root),
}
plan_path = project_dir / "results/frozen" / f"{run_name}_paths.json"
plan_path.parent.mkdir(parents=True, exist_ok=True)
plan_path.write_text(json.dumps(plan, indent=2))

print("Пути запуска:", plan_path, flush=True)

for part in ["train", "val", "test"]:
    out_dir = cache_root / part
    log_path = logs_dir / f"{run_name}_{part}.txt"

    command = [
        sys.executable, "-u", "-m", "routing.frozen.export",
        "--dataset", "cityscapes",
        "--root", str(data_root),
        "--split", "val",
        "--ids", str(split_dir / f"{part}.txt"),
        "--out", str(out_dir),
        "--layers", "F2,F3,F4,C",
        "--b2-input", "image",
        "--precision", "fp32",
        "--max-batch", "1",
        "--workers", "0",
    ]

    print(f"\n=== Экспорт {part}: {out_dir} ===", flush=True)

    with log_path.open("w") as log:
        process = subprocess.Popen(
            command,
            cwd=project_dir,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
        )

        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()

        returncode = process.wait()

    if returncode != 0:
        raise RuntimeError(
            f"Ошибка экспорта {part}, код {returncode}. Лог: {log_path}"
        )

print("\nВсе три кэша готовы:", cache_root)

Пути запуска: /home/jupyter/project/error-geometry-routing-frozen/results/frozen/routing_v1_20261025_132318_paths.json

=== Экспорт train: /home/jupyter/project/error-geometry-routing-frozen/results/frozen/routing_v1_20261025_132318/train ===
/home/jupyter/.local/lib/python3.10/site-packages/transformers/utils/hub.py:128: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
2026-10-25 13:23:23.646545: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-10-25 13:23:23.696384: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX51

In [16]:
from pathlib import Path
import json
import hashlib

project_dir = Path("/home/jupyter/project/error-geometry-routing-frozen")
cache_root = project_dir / "results/frozen/routing_v1_20261025_132318"

manifests = {}

for part in ["train", "val", "test"]:
    directory = cache_root / part
    meta = json.loads((directory / "manifest.json").read_text())
    manifests[part] = meta

    assert meta["complete"], f"Незавершённый кэш: {part}"

    missing = []
    for image_id in meta["images"]:
        filename = hashlib.sha256(image_id.encode()).hexdigest()[:24] + ".npz"
        if not (directory / filename).exists():
            missing.append(image_id)

    assert not missing, (part, missing[:5])
    print(part, "—", len(meta["images"]), "кадров, кэш готов")

assert not set(manifests["train"]["images"]) & set(manifests["val"]["images"])
assert not set(manifests["train"]["images"]) & set(manifests["test"]["images"])
assert not set(manifests["val"]["images"]) & set(manifests["test"]["images"])

train — 240 кадров, кэш готов
val — 60 кадров, кэш готов
test — 200 кадров, кэш готов


In [17]:
import subprocess
import sys

runs_root = cache_root / "selectors_v1"
runs_root.mkdir(exist_ok=True)

logs_dir = project_dir / "logs"
logs_dir.mkdir(exist_ok=True)

experiments = {
    "head_entropy": [],
    "head_f2_f3": ["--layers", "F2,F3"],
}

for name, extra_args in experiments.items():
    out_dir = runs_root / name

    if out_dir.exists():
        history_path = out_dir / "history.json"
        if (
            history_path.exists()
            and len(json.loads(history_path.read_text())) >= 5
            and (out_dir / "selector.pt").exists()
        ):
            print(name, "уже завершён — пропускаем")
            continue

        raise RuntimeError(
            f"Каталог {out_dir} уже существует, но запуск не завершён. "
            "Не перезаписываем его."
        )

    command = [
        sys.executable, "-u", "-m", "routing.frozen.train",
        "--train", str(cache_root / "train"),
        "--val", str(cache_root / "val"),
        "--out", str(out_dir),
        "--kind", "spatial",
        "--confidence", "entropy",
        "--epochs", "5",
        "--width", "16",
        "--lr", "0.001",
        "--seed", "0",
        "--device", "cuda",
        *extra_args,
    ]

    log_path = logs_dir / f"routing_v1_{name}_train.txt"
    print(f"\n=== Обучение {name} ===", flush=True)

    with log_path.open("w") as log:
        process = subprocess.Popen(
            command,
            cwd=project_dir,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
        )

        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()

        returncode = process.wait()

    if returncode != 0:
        raise RuntimeError(f"{name}: ошибка {returncode}. Лог: {log_path}")

print("\nОба селектора обучены:", runs_root)


=== Обучение head_entropy ===
{'epoch': 1, 'train_mse': 0.020474310767410013, 'val_mse': 0.016620539066692194}
{'epoch': 2, 'train_mse': 0.019160339162529755, 'val_mse': 0.01659436816892897}
{'epoch': 3, 'train_mse': 0.019133357919054107, 'val_mse': 0.016565245615007978}
{'epoch': 4, 'train_mse': 0.01911281398012458, 'val_mse': 0.01651977956450234}
{'epoch': 5, 'train_mse': 0.01908882047961621, 'val_mse': 0.016494433643917242}

=== Обучение head_f2_f3 ===
{'epoch': 1, 'train_mse': 0.02245429292379413, 'val_mse': 0.01694948412089919}
{'epoch': 2, 'train_mse': 0.019297953199323578, 'val_mse': 0.01666237899335101}
{'epoch': 3, 'train_mse': 0.01898251856910065, 'val_mse': 0.01647719513469686}
{'epoch': 4, 'train_mse': 0.018856163494638166, 'val_mse': 0.016457044150835525}
{'epoch': 5, 'train_mse': 0.018762592022540048, 'val_mse': 0.016457548330072312}

Оба селектора обучены: /home/jupyter/project/error-geometry-routing-frozen/results/frozen/routing_v1_20261025_132318/selectors_v1


In [18]:
for name in experiments:
    run_dir = runs_root / name
    history = json.loads((run_dir / "history.json").read_text())
    best = min(history, key=lambda row: row["val_mse"])

    print(
        name,
        "| лучшая эпоха:", best["epoch"],
        "| train MSE:", round(best["train_mse"], 6),
        "| val MSE:", round(best["val_mse"], 6),
        "| checkpoint:", (run_dir / "selector.pt").exists(),
    )

head_entropy | лучшая эпоха: 5 | train MSE: 0.019089 | val MSE: 0.016494 | checkpoint: True
head_f2_f3 | лучшая эпоха: 4 | train MSE: 0.018856 | val MSE: 0.016457 | checkpoint: True


In [19]:
from pathlib import Path
import subprocess
import sys
import json

project_dir = Path("/home/jupyter/project/error-geometry-routing-frozen")
cache_root = project_dir / "results/frozen/routing_v1_20261025_132318"
runs_root = cache_root / "selectors_v1"

evaluation_dir = runs_root / "validation_routing"
evaluation_dir.mkdir(exist_ok=True)

methods = {
    "raw_entropy": None,
    "head_entropy": runs_root / "head_entropy",
    "head_f2_f3": runs_root / "head_f2_f3",
}

for name, run_dir in methods.items():
    output_path = evaluation_dir / f"{name}.json"

    command = [
        sys.executable, "-u", "-m", "routing.frozen.calibrate",
        "--cache", str(cache_root / "val"),
        "--device", "cuda",
        "--quantiles", "0.5,0.6,0.7,0.8,0.9,0.95",
        "--out", str(output_path),
    ]

    if run_dir is None:
        command += ["--mode", "entropy"]
    else:
        command += ["--run", str(run_dir)]

    print(f"\n=== Validation: {name} ===", flush=True)

    result = subprocess.run(
        command,
        cwd=project_dir,
        capture_output=True,
        text=True,
    )

    (evaluation_dir / f"{name}.log").write_text(
        result.stdout + result.stderr,
        encoding="utf-8",
    )

    if result.returncode != 0:
        print(result.stdout)
        print(result.stderr)
        raise RuntimeError(f"Ошибка оценки {name}")

    print("Готово:", output_path)


=== Validation: raw_entropy ===
Готово: /home/jupyter/project/error-geometry-routing-frozen/results/frozen/routing_v1_20261025_132318/selectors_v1/validation_routing/raw_entropy.json

=== Validation: head_entropy ===
Готово: /home/jupyter/project/error-geometry-routing-frozen/results/frozen/routing_v1_20261025_132318/selectors_v1/validation_routing/head_entropy.json

=== Validation: head_f2_f3 ===
Готово: /home/jupyter/project/error-geometry-routing-frozen/results/frozen/routing_v1_20261025_132318/selectors_v1/validation_routing/head_f2_f3.json


In [20]:
import pandas as pd

rows = []

for name in methods:
    report = json.loads(
        (evaluation_dir / f"{name}.json").read_text()
    )

    for point in report["rows"]:
        rows.append({
            "метод": name,
            "порог": point["threshold"],
            "окон/кадр": point["mean_windows"],
            "mIoU": point["cached_miou"],
        })

table = pd.DataFrame(rows)

print(
    table.sort_values(["метод", "окон/кадр"])
    .round({"порог": 6, "окон/кадр": 3, "mIoU": 3})
    .to_string(index=False)
)

table.to_csv(
    evaluation_dir / "validation_routing.csv",
    index=False,
)

       метод     порог  окон/кадр   mIoU
head_entropy  0.070955      0.000 70.148
head_entropy  0.045452      0.400 71.159
head_entropy  0.035913      0.800 73.231
head_entropy  0.027217      1.600 75.048
head_entropy  0.020249      2.400 75.725
head_entropy  0.014733      3.200 76.306
head_entropy  0.008998      4.000 76.979
head_entropy -0.001963      7.983 77.626
  head_f2_f3  0.077969      0.000 70.148
  head_f2_f3  0.049664      0.400 72.189
  head_f2_f3  0.037937      0.800 73.768
  head_f2_f3  0.029389      1.600 74.930
  head_f2_f3  0.021502      2.400 76.280
  head_f2_f3  0.016126      3.200 76.349
  head_f2_f3  0.009671      4.000 76.938
  head_f2_f3 -0.010954      7.983 77.623
 raw_entropy  0.472426      0.000 70.148
 raw_entropy  0.307374      0.400 71.159
 raw_entropy  0.247544      0.800 73.238
 raw_entropy  0.191101      1.600 75.048
 raw_entropy  0.145136      2.400 75.744
 raw_entropy  0.109792      3.200 76.220
 raw_entropy  0.072631      4.000 76.979
 raw_entropy  0.

In [21]:
from pathlib import Path

project_dir = Path("/home/jupyter/project/error-geometry-routing-frozen")
source_path = project_dir / "routing/frozen/calibrate.py"

source = source_path.read_text()

old = (
    "cuts = [float(np.nextafter(values.max(), np.inf)), "
    "float(np.nextafter(values.min(), -np.inf))]"
)

new = (
    "cuts = [float(np.nextafter(values.max(), "
    "np.asarray(np.inf, dtype=values.dtype))),\n"
    "            float(np.nextafter(values.min(), "
    "np.asarray(-np.inf, dtype=values.dtype)))]"
)

if old in source:
    source_path.write_text(source.replace(old, new, 1))
    print("Исправлено сохранение точности крайних порогов")
elif new in source:
    print("Исправление уже применено")
else:
    raise RuntimeError("Код отличается — автоматическая замена не выполнена")

Исправлено сохранение точности крайних порогов


In [22]:
import json

cache_root = project_dir / "results/frozen/routing_v1_20261025_132318"
runs_root = cache_root / "selectors_v1"
evaluation_dir = runs_root / "validation_routing"
split_dir = project_dir / "splits/frozen_router_v1"

shortlists = {}

for name in ["raw_entropy", "head_f2_f3"]:
    report = json.loads((evaluation_dir / f"{name}.json").read_text())

    selected = []
    for target in [0.8, 2.4]:
        point = min(
            report["rows"],
            key=lambda row: abs(row["mean_windows"] - target),
        )
        if point not in selected:
            selected.append(point)

    report["rows"] = selected

    path = evaluation_dir / f"{name}_shortlist.json"
    path.write_text(json.dumps(report, indent=2))
    shortlists[name] = path

    print(name)
    for point in selected:
        print(point)

raw_entropy
{'threshold': 0.2475441634654999, 'cached_miou': 73.23843799958189, 'mean_windows': 0.8}
{'threshold': 0.14513574242591856, 'cached_miou': 75.74361216758362, 'mean_windows': 2.4}
head_f2_f3
{'threshold': 0.037937160581350333, 'cached_miou': 73.7676889466907, 'mean_windows': 0.8}
{'threshold': 0.021502112597227077, 'cached_miou': 76.27980719676292, 'mean_windows': 2.4}


In [23]:
import subprocess
import sys

jobs = [
    ("b0", ["--mode", "b0"]),
    ("b2_full", ["--mode", "b2"]),
    (
        "raw_entropy",
        [
            "--mode", "entropy",
            "--calibration", str(shortlists["raw_entropy"]),
            "--validation-sweep",
        ],
    ),
    (
        "head_f2_f3",
        [
            "--mode", "learned",
            "--run", str(runs_root / "head_f2_f3"),
            "--calibration", str(shortlists["head_f2_f3"]),
            "--validation-sweep",
        ],
    ),
]

live_dir = evaluation_dir / "live_fp32_v1"
live_dir.mkdir(exist_ok=True)

for name, extra in jobs:
    output_path = live_dir / f"{name}.json"

    if output_path.exists():
        print(name, "уже измерен — пропускаем")
        continue

    command = [
        sys.executable, "-u", "-m", "routing.frozen.live",
        "--dataset", "cityscapes",
        "--root", str(project_dir / "data/cityscapes"),
        "--split", "val",
        "--ids", str(split_dir / "val.txt"),
        "--b2-input", "image",
        "--precision", "fp32",
        "--max-batch", "1",
        "--warmup", "3",
        "--repeats", "2",
        "--workers", "0",
        "--out", str(output_path),
        *extra,
    ]

    print(f"\n=== Реальный замер: {name} ===", flush=True)

    with (live_dir / f"{name}.log").open("w") as log:
        process = subprocess.Popen(
            command,
            cwd=project_dir,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
        )

        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()

        returncode = process.wait()

    if returncode != 0:
        raise RuntimeError(f"Ошибка {name}; смотри {live_dir / f'{name}.log'}")

print("\nВсе замеры готовы:", live_dir)


=== Реальный замер: b0 ===
/home/jupyter/.local/lib/python3.10/site-packages/transformers/utils/hub.py:128: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
2026-10-25 14:10:14.176006: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-10-25 14:10:14.222603: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-10-25 14:10:15.016747: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
/u

In [24]:
import pandas as pd

rows = []

for name, _ in jobs:
    report = json.loads((live_dir / f"{name}.json").read_text())

    for point in report["rows"]:
        rows.append({
            "метод": name,
            "порог": point["threshold"],
            "окон/кадр": point["mean_windows"],
            "mIoU": point["miou"],
            "среднее мс": point["mean_ms"],
            "p95 мс": point["p95_ms"],
            "память MiB": point["peak_allocated_bytes"] / 1024**2,
        })

live_table = pd.DataFrame(rows)
print(live_table.round(3).to_string(index=False))
live_table.to_csv(live_dir / "summary.csv", index=False)

      метод  порог  окон/кадр   mIoU  среднее мс  p95 мс  память MiB
         b0    NaN        0.0 70.148      38.709  39.129     332.563
    b2_full    NaN        0.0 77.211     716.294 728.780    3662.880
raw_entropy  0.248        0.8 73.238     150.047 314.853    1147.006
raw_entropy  0.145        2.4 75.744     370.142 599.278    1147.006
 head_f2_f3  0.038        0.8 73.768     154.811 455.325    1152.278
 head_f2_f3  0.022        2.4 76.280     376.068 597.898    1152.278


In [25]:
from pathlib import Path
import json
import subprocess
import sys

project_dir = Path("/home/jupyter/project/error-geometry-routing-frozen")
cache_root = project_dir / "results/frozen/routing_v1_20261025_132318"
runs_root = cache_root / "selectors_v1"
evaluation_dir = runs_root / "validation_routing"
split_dir = project_dir / "splits/frozen_router_v1"

test_dir = runs_root / "test_live_fp32_v1"
test_dir.mkdir(exist_ok=True)

test_jobs = [
    ("b0", ["--mode", "b0"]),
    ("b2_full", ["--mode", "b2"]),
]

for name in ["raw_entropy", "head_f2_f3"]:
    report = json.loads(
        (evaluation_dir / f"{name}_shortlist.json").read_text()
    )

    for target in [0.8, 2.4]:
        point = min(
            report["rows"],
            key=lambda row: abs(row["mean_windows"] - target),
        )

        extra = [
            "--threshold", repr(point["threshold"]),
        ]

        if name == "raw_entropy":
            extra += ["--mode", "entropy"]
        else:
            extra += [
                "--mode", "learned",
                "--run", str(runs_root / "head_f2_f3"),
            ]

        test_jobs.append((f"{name}_val{target}", extra))

for name, extra in test_jobs:
    output_path = test_dir / f"{name}.json"

    if output_path.exists():
        print(name, "уже измерен — пропускаем")
        continue

    command = [
        sys.executable, "-u", "-m", "routing.frozen.live",
        "--dataset", "cityscapes",
        "--root", str(project_dir / "data/cityscapes"),
        "--split", "val",
        "--ids", str(split_dir / "test.txt"),
        "--b2-input", "image",
        "--precision", "fp32",
        "--max-batch", "1",
        "--warmup", "3",
        "--repeats", "2",
        "--workers", "0",
        "--out", str(output_path),
        *extra,
    ]

    print(f"\n=== P1: {name} ===", flush=True)

    with (test_dir / f"{name}.log").open("w") as log:
        process = subprocess.Popen(
            command,
            cwd=project_dir,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
        )

        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()

        returncode = process.wait()

    if returncode != 0:
        raise RuntimeError(f"Ошибка {name}; смотри лог в {test_dir}")

print("\nОценка P1 завершена:", test_dir)


=== P1: b0 ===
/home/jupyter/.local/lib/python3.10/site-packages/transformers/utils/hub.py:128: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
2026-10-25 14:23:13.191736: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-10-25 14:23:13.239312: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-10-25 14:23:14.042682: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
/usr/local/lib

In [27]:
import pandas as pd

rows = []

for name, _ in test_jobs:
    report = json.loads((test_dir / f"{name}.json").read_text())

    for point in report["rows"]:
        rows.append({
            "метод": name,
            "окон/кадр": point["mean_windows"],
            "mIoU": point["miou"],
            "среднее мс": point["mean_ms"],
            "p95 мс": point["p95_ms"],
            "память MiB": point["peak_allocated_bytes"] / 1024**2,
        })

test_table = pd.DataFrame(rows)
print(test_table.round(3).to_string(index=False))
test_table.to_csv(test_dir / "summary.csv", index=False)

             метод  окон/кадр   mIoU  среднее мс  p95 мс  память MiB
                b0      0.000 71.569      38.959  39.702     332.563
           b2_full      0.000 79.954     728.596 736.517    3662.880
raw_entropy_val0.8      0.975 76.471     175.097 454.569    1147.006
raw_entropy_val2.4      2.735 79.901     416.299 725.800    1147.006
 head_f2_f3_val0.8      0.955 76.683     175.948 457.713    1152.278
 head_f2_f3_val2.4      2.730 79.706     420.568 733.453    1152.278


In [28]:
from pathlib import Path
import json
import numpy as np

test_dir = Path(
    "/home/jupyter/project/error-geometry-routing-frozen/"
    "results/frozen/routing_v1_20261025_132318/"
    "selectors_v1/test_live_fp32_v1"
)

names = [
    "raw_entropy_val0.8",
    "head_f2_f3_val0.8",
    "raw_entropy_val2.4",
    "head_f2_f3_val2.4",
]

matrices = {}
reference_ids = None

for name in names:
    report = json.loads((test_dir / f"{name}.json").read_text())
    frames = report["rows"][0]["frames"]

    by_id = {frame["image_id"]: frame for frame in frames}
    assert len(by_id) == len(frames), "Повторяющиеся image_id"

    ids = sorted(by_id)

    if reference_ids is None:
        reference_ids = ids
    else:
        assert ids == reference_ids, "Наборы кадров различаются"

    matrices[name] = np.stack([
        np.asarray(by_id[i]["confusion"], dtype=np.float64)
        for i in ids
    ])


def batch_miou(conf):
    tp = np.diagonal(conf, axis1=-2, axis2=-1)
    union = conf.sum(axis=-2) + conf.sum(axis=-1) - tp

    iou = np.divide(
        tp, union,
        out=np.full_like(tp, np.nan),
        where=union > 0,
    )
    return 100 * np.nanmean(iou, axis=-1)


comparisons = [
    ("малый бюджет", "head_f2_f3_val0.8", "raw_entropy_val0.8"),
    ("основной бюджет", "head_f2_f3_val2.4", "raw_entropy_val2.4"),
]

results = []

for unit in ["кадр", "последовательность"]:
    group_ids = (
        reference_ids
        if unit == "кадр"
        else ["_".join(i.split("_")[:2]) for i in reference_ids]
    )

    unique_groups = sorted(set(group_ids))
    indices = {
        group: np.flatnonzero(np.asarray(group_ids) == group)
        for group in unique_groups
    }

    grouped = {
        name: np.stack([
            conf[indices[group]].sum(axis=0)
            for group in unique_groups
        ])
        for name, conf in matrices.items()
    }

    rng = np.random.default_rng(42)
    n = len(unique_groups)

    weights = rng.multinomial(
        n,
        np.full(n, 1 / n),
        size=2000,
    ).astype(np.float64)

    bootstrap_scores = {
        name: batch_miou(
            (weights @ conf.reshape(n, -1)).reshape(-1, 19, 19)
        )
        for name, conf in grouped.items()
    }

    for label, candidate, control in comparisons:
        delta = (
            batch_miou(matrices[candidate].sum(axis=0))
            - batch_miou(matrices[control].sum(axis=0))
        )

        distribution = (
            bootstrap_scores[candidate]
            - bootstrap_scores[control]
        )
        low, high = np.percentile(distribution, [2.5, 97.5])

        row = {
            "режим": label,
            "единица": unit,
            "число_групп": n,
            "delta_miou": float(delta),
            "ci_low": float(low),
            "ci_high": float(high),
        }
        results.append(row)

        print(
            f"{label}, {unit}: "
            f"Δ={delta:+.3f}, "
            f"95% CI [{low:+.3f}, {high:+.3f}], "
            f"групп={n}"
        )

(test_dir / "bootstrap_f2_f3_vs_entropy.json").write_text(
    json.dumps(results, ensure_ascii=False, indent=2)
)

малый бюджет, кадр: Δ=+0.212, 95% CI [-0.171, +0.537], групп=200
основной бюджет, кадр: Δ=-0.195, 95% CI [-0.637, +0.129], групп=200
малый бюджет, последовательность: Δ=+0.212, 95% CI [-0.145, +0.357], групп=94
основной бюджет, последовательность: Δ=-0.195, 95% CI [-1.851, +0.121], групп=94


811

In [29]:
from pathlib import Path
import json
import zipfile
import hashlib
import subprocess
import sys

project_dir = Path("/home/jupyter/project/error-geometry-routing-frozen")
cache_root = project_dir / "results/frozen/routing_v1_20261025_132318"
runs_root = cache_root / "selectors_v1"

export_dir = project_dir / "exports"
export_dir.mkdir(exist_ok=True)

archive_path = export_dir / "routing_v1_stage1.zip"

files = set()

# Веса, конфигурации, таблицы, сквозные замеры и bootstrap.
files.update(p for p in runs_root.rglob("*") if p.is_file())

# Разбиения и описания кэшей.
files.update((project_dir / "splits/frozen_router_v1").glob("*.txt"))
for part in ["train", "val", "test"]:
    files.add(cache_root / part / "manifest.json")

# Исходники для воспроизводимости.
for directory in ["routing/frozen", "core"]:
    files.update((project_dir / directory).rglob("*.py"))

for relative in [
    "tests/test_frozen_router.py",
    "docs/frozen_router_runbook.md",
]:
    path = project_dir / relative
    if path.exists():
        files.add(path)

files.update((project_dir / "logs").glob("routing_v1*.txt"))

freeze = subprocess.run(
    [sys.executable, "-m", "pip", "freeze"],
    capture_output=True,
    text=True,
    check=True,
)
environment_path = export_dir / "routing_v1_environment.txt"
environment_path.write_text(freeze.stdout)
files.add(environment_path)

if archive_path.exists():
    raise RuntimeError(
        "Архив уже существует. Не перезаписываем: "
        "скачай его или выбери новое имя."
    )

with zipfile.ZipFile(
    archive_path, "w", compression=zipfile.ZIP_DEFLATED
) as archive:
    for path in sorted(files):
        if path.is_file():
            archive.write(path, path.relative_to(project_dir))

digest = hashlib.sha256(archive_path.read_bytes()).hexdigest()
archive_path.with_suffix(".sha256").write_text(digest + "\n")

print("Архив:", archive_path)
print("Размер:", round(archive_path.stat().st_size / 1024**2, 1), "МиБ")
print("SHA256:", digest)

Архив: /home/jupyter/project/error-geometry-routing-frozen/exports/routing_v1_stage1.zip
Размер: 0.6 МиБ
SHA256: d298a6acb3bfaf0a29081b95bf68d9ca46d60201860dd43b1eedbae83e449063


In [4]:
from pathlib import Path
import sys
import subprocess
import json
import time
import os
import numpy as np

project_dir = Path("/home/jupyter/project/error-geometry-routing-frozen")
cache_root = project_dir / "results/frozen/routing_v1_20261025_132318"
runs_root = cache_root / "selectors_v1"

assert project_dir.exists(), f"Не найдена папка: {project_dir}"

for part in ["train", "val", "test"]:
    path = cache_root / part / "manifest.json"
    assert path.exists(), f"Не найден кэш: {path}"

print("Пути восстановлены, кэши найдены")

Пути восстановлены, кэши найдены


In [6]:
import time
import os

geometry_root = cache_root / "geometry_v1"
geometry_root.mkdir(exist_ok=True)

def run_logged(name, arguments):
    log_path = geometry_root / f"{name}.log"
    command = [sys.executable, "-u", "-m", *arguments]

    env = os.environ.copy()
    env["OMP_NUM_THREADS"] = "4"
    env["OPENBLAS_NUM_THREADS"] = "4"
    env["MKL_NUM_THREADS"] = "4"

    print(f"\n=== {name} ===", flush=True)
    started = time.perf_counter()

    with log_path.open("w") as log:
        process = subprocess.Popen(
            command,
            cwd=project_dir,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
        )

        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()

        code = process.wait()

    print(
        f"\n{name}: код {code}, "
        f"{(time.perf_counter() - started) / 60:.1f} мин",
        flush=True,
    )

    if code != 0:
        raise RuntimeError(f"Ошибка. Лог: {log_path}")

In [7]:
reference_dir = geometry_root / "reference"

if reference_dir.exists():
    raise RuntimeError(
        f"Каталог уже существует: {reference_dir}. "
        "Не перезаписываем опорную геометрию."
    )

run_logged(
    "fit_reference",
    [
        "routing.frozen.geometry",
        "--ref", str(cache_root / "train"),
        "--layers", "F2,F3",
        "--out", str(reference_dir),
        "--max-samples", "20000",
        "--per-image", "512",
        "--purity", "0.8",
        "--eta", "0.001",
        "--seed", "0",
    ],
)


=== fit_reference ===

fit_reference: код 0, 1.3 мин


In [8]:
import numpy as np

with np.load(reference_dir / "statistics.npz") as statistics:
    missing_by_layer = {}

    for layer in ["F2", "F3"]:
        present = statistics[f"{layer}__present"].astype(bool)
        missing = np.flatnonzero(~present).tolist()
        missing_by_layer[layer] = missing

        for key in ["mean", "scale", "centers", "whitening"]:
            assert np.isfinite(
                statistics[f"{layer}__{key}"]
            ).all(), (layer, key)

        print(
            layer,
            "| классов с опорами:", int(present.sum()),
            "| отсутствуют trainId:", missing,
        )

print("Статистики конечны: OK")

F2 | классов с опорами: 19 | отсутствуют trainId: []
F3 | классов с опорами: 19 | отсутствуют trainId: []
Статистики конечны: OK


In [13]:
selector_jobs = {
    "ridge_base": [],
    "euclidean_F2": [
        "--layers", "F2",
        "--geometry", str(reference_dir),
        "--metric", "euclidean",
    ],
    "euclidean_F3": [
        "--layers", "F3",
        "--geometry", str(reference_dir),
        "--metric", "euclidean",
    ],
    "mahalanobis_F2": [
        "--layers", "F2",
        "--geometry", str(reference_dir),
        "--metric", "mahalanobis",
    ],
    "mahalanobis_F3": [
        "--layers", "F3",
        "--geometry", str(reference_dir),
        "--metric", "mahalanobis",
    ],
}

for name, extra in selector_jobs.items():
    out_dir = geometry_root / name

    if out_dir.exists():
        if all(
            (out_dir / filename).exists()
            for filename in ["selector.npz", "config.json", "history.json"]
        ):
            print(name, "уже подготовлен — пропускаем")
            continue

        raise RuntimeError(f"Незавершённый каталог: {out_dir}")

    run_logged(
        f"train_{name}",
        [
            "routing.frozen.train",
            "--train", str(cache_root / "train"),
            "--val", str(cache_root / "val"),
            "--out", str(out_dir),
            "--kind", "ridge",
            "--confidence", "all",
            "--alphas", "0.001,0.01,0.1,1",
            "--device", "cpu",
            "--seed", "0",
            *extra,
        ],
    )


=== train_ridge_base ===
[{'alpha': 0.001, 'val_mse': 0.004078724342402365}, {'alpha': 0.01, 'val_mse': 0.0040798134393653564}, {'alpha': 0.1, 'val_mse': 0.0040685066773928375}, {'alpha': 1.0, 'val_mse': 0.0040801410111175555}]

train_ridge_base: код 0, 1.7 мин

=== train_euclidean_F2 ===
[{'alpha': 0.001, 'val_mse': 0.004082981992179732}, {'alpha': 0.01, 'val_mse': 0.004083340996834867}, {'alpha': 0.1, 'val_mse': 0.004072618557518604}, {'alpha': 1.0, 'val_mse': 0.004081608428393049}]

train_euclidean_F2: код 0, 5.1 мин

=== train_euclidean_F3 ===
[{'alpha': 0.001, 'val_mse': 0.0040742201277525515}, {'alpha': 0.01, 'val_mse': 0.0040781935212069005}, {'alpha': 0.1, 'val_mse': 0.004069860309135135}, {'alpha': 1.0, 'val_mse': 0.004077055685310504}]

train_euclidean_F3: код 0, 4.9 мин

=== train_mahalanobis_F2 ===
[{'alpha': 0.001, 'val_mse': 0.004090807546389303}, {'alpha': 0.01, 'val_mse': 0.0040908506873415795}, {'alpha': 0.1, 'val_mse': 0.004078608789685363}, {'alpha': 1.0, 'val_mse':

In [10]:
from pathlib import Path

project_dir = Path("/home/jupyter/project/error-geometry-routing-frozen")
path = project_dir / "routing/frozen/regional.py"
source = path.read_text()

old = '''def resize_map(x, shape):
    return ndimage.zoom(x, (shape[0] / x.shape[0], shape[1] / x.shape[1]), order=1, prefilter=False)'''

new = '''def resize_map(x, shape):
    x = np.asarray(x)
    if x.dtype == np.float16:
        x = x.astype(np.float32)
    return ndimage.zoom(x, (shape[0] / x.shape[0], shape[1] / x.shape[1]), order=1, prefilter=False)'''

if old in source:
    path.write_text(source.replace(old, new, 1))
    print("Исправлено")
elif new in source:
    print("Исправление уже применено")
else:
    raise RuntimeError("Функция отличается — автоматическая замена остановлена")

Исправлено


In [11]:
import subprocess
import sys

check = subprocess.run(
    [
        sys.executable, "-c",
        "import numpy as np; "
        "from routing.frozen.regional import resize_map; "
        "x = resize_map(np.ones((2, 2), dtype=np.float16), (8, 8)); "
        "assert x.shape == (8, 8); "
        "assert np.isfinite(x).all(); "
        "print('Интерполяция float16 → float32: OK')"
    ],
    cwd=project_dir,
    capture_output=True,
    text=True,
)

print(check.stdout)
print(check.stderr)
assert check.returncode == 0

Интерполяция float16 → float32: OK




In [12]:
from datetime import datetime

failed_dir = geometry_root / "ridge_base"

if failed_dir.exists():
    assert not (failed_dir / "selector.npz").exists(), (
        "Есть сохранённый селектор — каталог не переименован"
    )

    backup = failed_dir.with_name(
        f"ridge_base_failed_{datetime.now():%Y%m%d_%H%M%S}"
    )
    failed_dir.rename(backup)
    print("Незавершённый запуск сохранён:", backup)

Незавершённый запуск сохранён: /home/jupyter/project/error-geometry-routing-frozen/results/frozen/routing_v1_20261025_132318/geometry_v1/ridge_base_failed_20261025_153154


In [14]:
import json

for name in selector_jobs:
    run_dir = geometry_root / name

    config = json.loads((run_dir / "config.json").read_text())
    history = json.loads((run_dir / "history.json").read_text())
    best = min(history, key=lambda row: row["val_mse"])

    print(
        name,
        "| alpha:", config["selected_alpha"],
        "| val MSE:", round(best["val_mse"], 7),
    )

ridge_base | alpha: 0.1 | val MSE: 0.0040685
euclidean_F2 | alpha: 0.1 | val MSE: 0.0040726
euclidean_F3 | alpha: 0.1 | val MSE: 0.0040699
mahalanobis_F2 | alpha: 0.1 | val MSE: 0.0040786
mahalanobis_F3 | alpha: 1.0 | val MSE: 0.0040777


In [15]:
for name in selector_jobs:
    output_path = geometry_root / name / "validation_routing.json"

    if output_path.exists():
        report = json.loads(output_path.read_text())
        assert report.get("rows"), f"Пустой отчёт: {output_path}"
        print(name, "уже оценён — пропускаем")
        continue

    run_logged(
        f"validate_{name}",
        [
            "routing.frozen.calibrate",
            "--cache", str(cache_root / "val"),
            "--run", str(geometry_root / name),
            "--device", "cpu",
            "--quantiles", "0.5,0.6,0.7,0.8,0.9,0.95",
            "--out", str(output_path),
        ],
    )

print("Оценка всех пяти методов завершена")


=== validate_ridge_base ===
[{'threshold': 0.06563068666872206, 'cached_miou': 70.14762545127333, 'mean_windows': 0.0}, {'threshold': 0.03916347004599764, 'cached_miou': 72.09361030049246, 'mean_windows': 0.4}, {'threshold': 0.03206233686670957, 'cached_miou': 73.26754328075964, 'mean_windows': 0.8}, {'threshold': 0.022953392923311285, 'cached_miou': 75.25526395882383, 'mean_windows': 1.6}, {'threshold': 0.01765592715576033, 'cached_miou': 75.62718396323983, 'mean_windows': 2.4}, {'threshold': 0.012730496596448397, 'cached_miou': 76.49848826620314, 'mean_windows': 3.2}, {'threshold': 0.009158089569422732, 'cached_miou': 77.00845825971791, 'mean_windows': 4.0}, {'threshold': -0.0038509942629043536, 'cached_miou': 77.62573545966107, 'mean_windows': 8.0}]

validate_ridge_base: код 0, 0.4 мин

=== validate_euclidean_F2 ===
[{'threshold': 0.06706169399190383, 'cached_miou': 70.14762545127333, 'mean_windows': 0.0}, {'threshold': 0.04027777919797645, 'cached_miou': 71.13710716672729, 'mean_w

In [16]:
import numpy as np
import pandas as pd

rows = []

for name in selector_jobs:
    report = json.loads(
        (geometry_root / name / "validation_routing.json").read_text()
    )

    for point in report["rows"]:
        rows.append({
            "метод": name,
            "порог": point["threshold"],
            "окон/кадр": point["mean_windows"],
            "mIoU": point["cached_miou"],
        })

table = pd.DataFrame(rows)

# Полная таблица сохраняется на диск.
table.to_csv(
    geometry_root / "validation_summary.csv",
    index=False,
)

# В выводе — две заранее выбранные области бюджета.
selected = table[
    np.isclose(table["окон/кадр"], 0.8)
    | np.isclose(table["окон/кадр"], 2.4)
].copy()

base = selected[selected["метод"] == "ridge_base"][
    ["окон/кадр", "mIoU"]
].rename(columns={"mIoU": "mIoU контроля"})

selected = selected.merge(
    base,
    on="окон/кадр",
    how="left",
    validate="many_to_one",
)

selected["Δ к ridge_base"] = (
    selected["mIoU"] - selected["mIoU контроля"]
)

print(
    selected.sort_values(["окон/кадр", "метод"])
    .round(4)
    .to_string(index=False)
)

         метод  порог  окон/кадр    mIoU  mIoU контроля  Δ к ridge_base
  euclidean_F2 0.0317        0.8 73.0743        73.2675         -0.1933
  euclidean_F3 0.0318        0.8 73.0926        73.2675         -0.1750
mahalanobis_F2 0.0315        0.8 73.0787        73.2675         -0.1889
mahalanobis_F3 0.0307        0.8 73.1251        73.2675         -0.1425
    ridge_base 0.0321        0.8 73.2675        73.2675          0.0000
  euclidean_F2 0.0184        2.4 75.6970        75.6272          0.0699
  euclidean_F3 0.0182        2.4 75.5690        75.6272         -0.0581
mahalanobis_F2 0.0186        2.4 75.6776        75.6272          0.0504
mahalanobis_F3 0.0191        2.4 76.1982        75.6272          0.5710
    ridge_base 0.0177        2.4 75.6272        75.6272          0.0000


In [17]:
from pathlib import Path
import os
import sys
import json
import time
import hashlib
import subprocess

import numpy as np
import pandas as pd

project_dir = Path(
    "/home/jupyter/project/error-geometry-routing-frozen"
)
cache_root = (
    project_dir / "results/frozen/routing_v1_20261025_132318"
)
geometry_root = cache_root / "geometry_v1"

experiment_dir = geometry_root / "p1_finalists_v1"
experiment_dir.mkdir(exist_ok=True)

def read_json(path):
    return json.loads(Path(path).read_text())

def save_json(path, data):
    Path(path).write_text(
        json.dumps(data, ensure_ascii=False, indent=2)
    )

print("Результаты:", experiment_dir)

Результаты: /home/jupyter/project/error-geometry-routing-frozen/results/frozen/routing_v1_20261025_132318/geometry_v1/p1_finalists_v1


In [18]:
plan_path = experiment_dir / "plan.json"

if plan_path.exists():
    plan = read_json(plan_path)
    print("Используем ранее зафиксированный план")
else:
    calibration_paths = {
        "ridge_base":
            geometry_root / "ridge_base/validation_routing.json",
        "mahalanobis_F3":
            geometry_root / "mahalanobis_F3/validation_routing.json",
        "raw_entropy":
            cache_root / "selectors_v1/validation_routing/raw_entropy.json",
    }

    test_manifest = read_json(cache_root / "test/manifest.json")
    test_ids = test_manifest["images"]
    jobs = {}
    hashes = {}

    for name, calibration_path in calibration_paths.items():
        calibration = read_json(calibration_path)

        matches = [
            row for row in calibration["rows"]
            if np.isclose(row["mean_windows"], 2.4)
        ]
        assert len(matches) == 1, (name, matches)
        row = matches[0]

        validation_ids = calibration["validation"]["images"]
        assert not set(validation_ids) & set(test_ids)

        run = (
            geometry_root / name
            if name != "raw_entropy" else None
        )

        jobs[name] = {
            "mode": "learned" if run else "entropy",
            "run": str(run) if run else None,
            "threshold": row["threshold"],
            "validation_miou": row["cached_miou"],
            "validation_mean_windows": row["mean_windows"],
        }

        files = [calibration_path]
        if run:
            files += [
                run / "config.json",
                run / "history.json",
                run / "selector.npz",
            ]
            if name == "mahalanobis_F3":
                files += [
                    run / "geometry.npz",
                    run / "geometry_config.json",
                ]
            train_ids = read_json(
                run / "config.json"
            )["train_manifest"]["images"]
            assert not set(train_ids) & set(test_ids)

        for path in files:
            hashes[str(path)] = hashlib.sha256(
                path.read_bytes()
            ).hexdigest()

    # Код также фиксируем контрольными суммами.
    for path in sorted((project_dir / "routing/frozen").glob("*.py")):
        hashes[str(path)] = hashlib.sha256(
            path.read_bytes()
        ).hexdigest()

    ids_path = experiment_dir / "p1_ids.txt"
    ids_path.write_text("\n".join(test_ids) + "\n")
    hashes[str(ids_path)] = hashlib.sha256(
        ids_path.read_bytes()
    ).hexdigest()

    plan = {
        "note": "Exploratory P1 evaluation; thresholds fixed on validation",
        "ids": str(ids_path),
        "jobs": jobs,
        "hashes": hashes,
    }
    save_json(plan_path, plan)

# При повторном запуске проверяем, что файлы не изменились.
for filename, expected in plan["hashes"].items():
    actual = hashlib.sha256(Path(filename).read_bytes()).hexdigest()
    assert actual == expected, f"Изменился файл: {filename}"

for name, job in plan["jobs"].items():
    print(
        name,
        "| threshold:", repr(job["threshold"]),
        "| validation mIoU:", round(job["validation_miou"], 4),
    )

ridge_base | threshold: 0.01765592715576033 | validation mIoU: 75.6272
mahalanobis_F3 | threshold: 0.019118412745518724 | validation mIoU: 76.1982
raw_entropy | threshold: 0.14513574242591856 | validation mIoU: 75.7436


In [10]:
import torch

assert torch.cuda.is_available(), "Переключи DataSphere на GPU"
print("GPU:", torch.cuda.get_device_name(0))

meta = read_json(cache_root / "test/manifest.json")
assert meta["precision"] == "fp32"

def run_live(name, job):
    output = experiment_dir / f"{name}.json"
    log_path = experiment_dir / f"{name}.log"

    if output.exists():
        result = read_json(output)
        assert len(result["rows"]) == 1
        assert {
            frame["image_id"]
            for frame in result["rows"][0]["frames"]
        } == set(Path(plan["ids"]).read_text().split())
        print(name, "уже готов — пропускаем")
        return

    command = [
        sys.executable, "-u", "-m", "routing.frozen.live",
        "--dataset", meta["dataset"],
        "--root", str(project_dir / "data/cityscapes"),
        # P1 — подвыборка размеченного Cityscapes val.
        "--split", "val",
        "--ids", plan["ids"],
        "--mode", job["mode"],
        "--threshold", repr(job["threshold"]),
        "--b0", meta["b0"],
        "--b2", meta["b2"],
        "--b2-input", meta["b2_input"],
        "--tile", str(meta["tile"]),
        "--margin", str(meta["margin"]),
        "--precision", "fp32",
        "--max-batch", "2",
        "--warmup", "3",
        "--repeats", "2",
        "--workers", "2",
        "--device", "cuda",
        "--out", str(output),
    ]
    if job["run"]:
        command += ["--run", job["run"]]

    env = os.environ.copy()
    for key in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"]:
        env[key] = "4"

    print(f"\n=== P1: {name} ===", flush=True)
    started = time.perf_counter()
    last_message = started

    with log_path.open("w") as log:
        process = subprocess.Popen(
            command,
            cwd=project_dir,
            env=env,
            stdout=log,
            stderr=subprocess.STDOUT,
        )
        while process.poll() is None:
            time.sleep(5)
            now = time.perf_counter()
            if now - last_message >= 30:
                print(
                    f"{name}: процесс работает, "
                    f"прошло {(now-started)/60:.1f} мин",
                    flush=True,
                )
                last_message = now

    if process.returncode != 0:
        print(log_path.read_text()[-12000:])
        raise RuntimeError(f"Ошибка. Лог: {log_path}")

    row = read_json(output)["rows"][0]
    print({k: v for k, v in row.items() if k != "frames"})
    print(f"Завершено за {(time.perf_counter()-started)/60:.1f} мин")

for name, job in plan["jobs"].items():
    run_live(name, job)

GPU: Tesla T4

=== P1: ridge_base ===
ridge_base: процесс работает, прошло 0.5 мин
ridge_base: процесс работает, прошло 1.0 мин
ridge_base: процесс работает, прошло 1.5 мин
ridge_base: процесс работает, прошло 2.0 мин
ridge_base: процесс работает, прошло 2.5 мин
ridge_base: процесс работает, прошло 3.0 мин
ridge_base: процесс работает, прошло 3.5 мин
ridge_base: процесс работает, прошло 4.0 мин
ridge_base: процесс работает, прошло 4.5 мин
ridge_base: процесс работает, прошло 5.0 мин
ridge_base: процесс работает, прошло 5.5 мин
ridge_base: процесс работает, прошло 6.0 мин
ridge_base: процесс работает, прошло 6.5 мин
ridge_base: процесс работает, прошло 7.0 мин
ridge_base: процесс работает, прошло 7.5 мин
{'threshold': 0.01765592715576033, 'miou': 79.79099734501543, 'mean_ms': 462.82896806499593, 'median_ms': 493.2326632497279, 'p95_ms': 736.2267708752825, 'mean_windows': 2.825, 'peak_allocated_bytes': 2384631808}
Завершено за 7.9 мин

=== P1: mahalanobis_F3 ===
mahalanobis_F3: процесс р

In [5]:
from pathlib import Path
import os
import sys
import json
import time
import hashlib
import subprocess

import numpy as np
import pandas as pd

project_dir = Path(
    "/home/jupyter/project/error-geometry-routing-frozen"
)
cache_root = (
    project_dir / "results/frozen/routing_v1_20261025_132318"
)
geometry_root = cache_root / "geometry_v1"
experiment_dir = geometry_root / "p1_finalists_v1"

def read_json(path):
    return json.loads(Path(path).read_text())

def save_json(path, data):
    Path(path).write_text(
        json.dumps(data, ensure_ascii=False, indent=2)
    )

plan = read_json(experiment_dir / "plan.json")

for filename, expected in plan["hashes"].items():
    actual = hashlib.sha256(Path(filename).read_bytes()).hexdigest()
    assert actual == expected, f"Изменился файл: {filename}"

print("План загружен, контрольные суммы совпадают.")

План загружен, контрольные суммы совпадают.


In [6]:
results = {
    name: read_json(experiment_dir / f"{name}.json")
    for name in plan["jobs"]
}

table = pd.DataFrame([
    {
        "метод": name,
        "окон/кадр": result["rows"][0]["mean_windows"],
        "mIoU": result["rows"][0]["miou"],
        "среднее мс": result["rows"][0]["mean_ms"],
        "p95 мс": result["rows"][0]["p95_ms"],
        "память MiB":
            result["rows"][0]["peak_allocated_bytes"] / 2**20,
    }
    for name, result in results.items()
])

table.to_csv(
    experiment_dir / "summary.csv", index=False
)
print(table.round(3).to_string(index=False))

         метод  окон/кадр   mIoU  среднее мс   p95 мс  память MiB
    ridge_base      2.825 79.791     463.383  738.754    2274.162
mahalanobis_F3      2.715 79.674     965.114 1345.211    2277.506
   raw_entropy      2.735 79.901     380.865  667.050    2274.506


In [7]:
sys.path.insert(0, str(project_dir))
from routing.frozen.common import miou

frame_maps = {
    name: {
        frame["image_id"]: np.asarray(
            frame["confusion"], dtype=np.int64
        )
        for frame in result["rows"][0]["frames"]
    }
    for name, result in results.items()
}

ids = sorted(frame_maps["ridge_base"])
for mapping in frame_maps.values():
    assert set(mapping) == set(ids)

def paired_bootstrap(candidate, control, by_sequence, repeats=2000):
    group_members = {}
    for image_id in ids:
        # Cityscapes: city_sequence_frame.
        group = (
            "_".join(image_id.split("_")[:2])
            if by_sequence else image_id
        )
        group_members.setdefault(group, []).append(image_id)

    groups = sorted(group_members)

    def grouped_confusions(name):
        return np.stack([
            np.stack([
                frame_maps[name][image_id]
                for image_id in group_members[group]
            ]).sum(axis=0)
            for group in groups
        ])

    a = grouped_confusions(candidate)
    b = grouped_confusions(control)

    delta = float(miou(a.sum(axis=0)) - miou(b.sum(axis=0)))
    rng = np.random.default_rng(42)
    samples = np.empty(repeats)

    for i in range(repeats):
        chosen = rng.integers(0, len(groups), size=len(groups))
        samples[i] = (
            miou(a[chosen].sum(axis=0))
            - miou(b[chosen].sum(axis=0))
        )

    low, high = np.percentile(samples, [2.5, 97.5])
    return {
        "candidate": candidate,
        "control": control,
        "grouping": "sequence" if by_sequence else "frame",
        "groups": len(groups),
        "delta_pp": delta,
        "ci95_pp": [float(low), float(high)],
        "repeats": repeats,
        "seed": 42,
    }

bootstrap_results = []

for control in ["ridge_base", "raw_entropy"]:
    for by_sequence in [False, True]:
        result = paired_bootstrap(
            "mahalanobis_F3", control, by_sequence
        )
        bootstrap_results.append(result)
        low, high = result["ci95_pp"]
        print(
            f"F3 против {control}, {result['grouping']}: "
            f"Δ={result['delta_pp']:+.3f} п.п.; "
            f"95% CI [{low:+.3f}, {high:+.3f}]; "
            f"групп={result['groups']}"
        )

save_json(
    experiment_dir / "bootstrap.json",
    bootstrap_results,
)

F3 против ridge_base, frame: Δ=-0.117 п.п.; 95% CI [-0.339, +0.069]; групп=200
F3 против ridge_base, sequence: Δ=-0.117 п.п.; 95% CI [-0.468, +0.136]; групп=94
F3 против raw_entropy, frame: Δ=-0.227 п.п.; 95% CI [-0.570, +0.010]; групп=200
F3 против raw_entropy, sequence: Δ=-0.227 п.п.; 95% CI [-1.535, -0.050]; групп=94


In [8]:
result = read_json(experiment_dir / "mahalanobis_F3.json")
row = result["rows"][0]

expected = set(Path(plan["ids"]).read_text().split())
actual = [frame["image_id"] for frame in row["frames"]]

print("Ожидалось кадров:", len(expected))
print("Сохранено кадров:", len(actual))
print("Все кадры есть:", set(actual) == expected)
print("Нет повторов:", len(actual) == len(set(actual)))
print("GPU:", result["gpu"])
print("Среднее время, мс:", row["mean_ms"])

Ожидалось кадров: 200
Сохранено кадров: 200
Все кадры есть: True
Нет повторов: True
GPU: Tesla T4
Среднее время, мс: 965.1143436800018


In [9]:
from pathlib import Path
from datetime import datetime
import os
import sys
import json
import time
import hashlib
import subprocess

import numpy as np
import pandas as pd
import torch

assert torch.cuda.is_available(), "Нужен GPU"
print("GPU:", torch.cuda.get_device_name(0))

project_dir = Path(
    "/home/jupyter/project/error-geometry-routing-frozen"
)
cache_root = (
    project_dir / "results/frozen/routing_v1_20261025_132318"
)
geometry_root = cache_root / "geometry_v1"
previous_dir = geometry_root / "p1_finalists_v1"

def read_json(path):
    return json.loads(Path(path).read_text())

def save_json(path, data):
    Path(path).write_text(
        json.dumps(data, ensure_ascii=False, indent=2)
    )

# Используем прежний план без изменения порогов.
plan = read_json(previous_dir / "plan.json")

for filename, expected in plan["hashes"].items():
    actual = hashlib.sha256(Path(filename).read_bytes()).hexdigest()
    assert actual == expected, f"Изменился файл: {filename}"

experiment_dir = geometry_root / (
    "p1_repeat_" + datetime.now().strftime("%Y%m%d_%H%M%S")
)
experiment_dir.mkdir()
save_json(experiment_dir / "plan.json", plan)

print("Предыдущие результаты:", previous_dir)
print("Повторный замер:", experiment_dir)

GPU: Tesla T4
Предыдущие результаты: /home/jupyter/project/error-geometry-routing-frozen/results/frozen/routing_v1_20261025_132318/geometry_v1/p1_finalists_v1
Повторный замер: /home/jupyter/project/error-geometry-routing-frozen/results/frozen/routing_v1_20261025_132318/geometry_v1/p1_repeat_20261025_183941


In [11]:
import json
from pathlib import Path
import numpy as np
import pandas as pd

geometry_root = Path(
    "/home/jupyter/project/error-geometry-routing-frozen/"
    "results/frozen/routing_v1_20261025_132318/geometry_v1"
)
previous_dir = geometry_root / "p1_finalists_v1"

# Находим последний повтор, в котором завершены все три метода.
names = ["ridge_base", "mahalanobis_F3", "raw_entropy"]
completed = [
    folder
    for folder in geometry_root.glob("p1_repeat_*")
    if folder.is_dir()
    and all((folder / f"{name}.json").exists() for name in names)
]
assert completed, "Не найдена папка с тремя завершёнными замерами"
repeat_dir = max(completed, key=lambda folder: folder.name)
print("Сравниваем с:", repeat_dir)

def read_result(path):
    return json.loads(path.read_text())["rows"][0]

rows = []

for name in names:
    old = read_result(previous_dir / f"{name}.json")
    new = read_result(repeat_dir / f"{name}.json")

    old_frames = {f["image_id"]: f for f in old["frames"]}
    new_frames = {f["image_id"]: f for f in new["frames"]}

    assert len(old["frames"]) == len(new["frames"]) == 200
    assert len(old_frames) == len(new_frames) == 200
    assert set(old_frames) == set(new_frames)

    rows.append({
        "метод": name,
        "mIoU ранее": old["miou"],
        "mIoU повтор": new["miou"],
        "Δ mIoU": new["miou"] - old["miou"],
        "мс ранее": old["mean_ms"],
        "мс повтор": new["mean_ms"],
        "Δ времени, %": 100 * (new["mean_ms"] / old["mean_ms"] - 1),
        "p95 повтор": new["p95_ms"],
        "окон/кадр повтор": new["mean_windows"],
        "кадров с другой матрицей ошибок": sum(
            not np.array_equal(
                old_frames[key]["confusion"],
                new_frames[key]["confusion"],
            )
            for key in old_frames
        ),
        "кадров с другим числом окон": sum(
            old_frames[key]["windows"] != new_frames[key]["windows"]
            for key in old_frames
        ),
    })

comparison = pd.DataFrame(rows)
output_path = repeat_dir / "repeat_comparison.csv"
comparison.to_csv(output_path, index=False)

print(comparison.round(4).to_string(index=False))
print("\nСохранено:", output_path)

Сравниваем с: /home/jupyter/project/error-geometry-routing-frozen/results/frozen/routing_v1_20261025_132318/geometry_v1/p1_repeat_20261025_183941
         метод  mIoU ранее  mIoU повтор  Δ mIoU  мс ранее  мс повтор  Δ времени, %  p95 повтор  окон/кадр повтор  кадров с другой матрицей ошибок  кадров с другим числом окон
    ridge_base     79.7910      79.7910     0.0  463.3833   462.8290       -0.1196    736.2268             2.825                                0                            0
mahalanobis_F3     79.6743      79.6743     0.0  965.1143   967.3994        0.2368   1348.4868             2.715                                0                            0
   raw_entropy     79.9009      79.9009     0.0  380.8648   380.9597        0.0249    666.9707             2.735                                0                            0

Сохранено: /home/jupyter/project/error-geometry-routing-frozen/results/frozen/routing_v1_20261025_132318/geometry_v1/p1_repeat_20261025_183941/repeat_com